## Settings & imports

In [1]:
import numpy as np
import pandas as pd
import os
import torch
import matplotlib.pyplot as plt
import string
from torch import nn
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import csv

In [2]:
import json

with open('../config.json', 'r') as f:
    config = json.load(f)

which_dataset = config["which_dataset"]
    
preprocessing_method = config["preprocessing_method"] # normalization / logarithm
remove_outer = config["remove_outer"]
how_many_outer_to_remove = config["how_many_outer_to_remove"]
elements_to_keep = config["elements_to_keep"]

target_path = config["target_path"][which_dataset]
results_path = config["results_path"]
figures_path = config["figures_path"]
saved_models_path = config["saved_models_path"][which_dataset]

In [3]:
columns_to_keep_inds = [el + '_i' for el in elements_to_keep]
columns_to_keep_inks = [el + '_a' for el in elements_to_keep]

## Loading the data

In [4]:
if which_dataset == 'old':
    inds_df = pd.read_csv(target_path, usecols = elements_to_keep + ['probka'])
    inds_df.rename(columns={"probka": "Sample_id"}, inplace=True)
elif which_dataset == 'new':
    inds_df = pd.read_csv(target_path, usecols = elements_to_keep + ['name'])
    inds_df['Sample_id'] = inds_df['name'].apply(lambda x: x.split('_')[0] if len(x.split('_'))==2 else x.split('_')[0] + x.split('_')[1])
    inds_df.drop(columns=['name'], inplace=True)

In [5]:
inds_df

,Al,S,Cr,Mn,Co,Cu,Zn,Pb,Sample_id
0,128.802194,45.653763,2.409124,1799.781253,56.690235,502.336893,2870.916590,22.130476,APP.0
1,116.257792,36.722882,1.058379,1581.439596,57.348903,464.263129,2601.675466,23.784549,APP.0
2,122.697569,35.889411,0.532113,1428.396392,50.187486,432.960761,2282.392424,21.017099,APP.0
3,115.074501,30.669791,0.000000,1320.925303,45.969427,401.713436,2029.819336,19.378964,APP.0
4,106.341451,28.020798,0.514569,1113.528551,43.902105,367.568736,1819.739444,22.273592,APP.0
...,...,...,...,...,...,...,...,...,...
5164,23.097790,0.878231,0.252606,23.427704,9.868429,34.810332,72.329800,12.207351,ML9C
5165,13.786563,0.273938,0.715716,12.530392,5.208374,25.741688,39.738312,5.391881,ML9C
5166,14.047676,0.000000,0.000000,8.181281,5.819657,17.626538,27.548466,6.966688,ML9C
5167,13.211118,0.143252,0.336808,8.534130,4.276628,18.799328,26.533838,4.294322,ML9C


## Preprocessing

In [6]:
inds_df = inds_df.reset_index(drop=True)

### Removing some data

1. Let's remove 'outer' samples.

In [7]:
def remove_outer(group, n):
    return group.iloc[n:-n] if len(group) > 2*n else pd.DataFrame(columns=group.columns)

In [8]:
inds_df = inds_df.groupby('Sample_id', group_keys=False).apply(remove_outer, n=how_many_outer_to_remove)

/tmp/ipykernel_23226/1842499225.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  inds_df = inds_df.groupby('Sample_id', group_keys=False).apply(remove_outer, n=how_many_outer_to_remove)


2. Let's keep only columns that we need.

To reduce the set of used elements run cell below. Then, instead of predicting 29 numbers, we will predict only 8. We will also use only 8 numbers as input.

In [9]:
inds_df = inds_df[elements_to_keep]

2. Let's remove rows with missing values.

In [10]:
(inds_df.shape[0] - inds_df.dropna().shape[0])/inds_df.shape[0]

0.0

In [11]:
inds_df.dropna(inplace=True)

In [12]:
X = np.array(inds_df.values)

In [13]:
######################################################

### Normalizing / taking logarithm

In [14]:
def adjusted_log_transform(input_array):
    res = np.where(input_array>0, np.log(input_array), 0.)
    #res = np.where(input_array<=0, input_array + res.min(axis=0), res)
    return res

In [15]:
if preprocessing_method == 'normalization':

    X = (X - np.min(X, axis=0))/np.std(X, axis=0)
    
elif preprocessing_method == 'logarithm':
    
    X = adjusted_log_transform(X)

elif preprocessing_method == 'logarithm_and_normalization':
    
    #logarithm
    X = adjusted_log_transform(X)
    
    #normalization
    X = (X - np.min(X, axis=0))/np.std(X, axis=0)

elif preprocessing_method == 'none':
    
    pass

/tmp/ipykernel_23226/3365076432.py:2: RuntimeWarning: divide by zero encountered in log
  res = np.where(input_array>0, np.log(input_array), 0.)


In [16]:
# coor = 0
# plt.hist(X[:,coor], bins=100)

### Converting to tensors

In [17]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cpu device


/home/basia/myvirtualenv/lib/python3.12/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [18]:
X = torch.Tensor(X).to(device)

## Loading the trained model

### Model class

In [19]:
input_size = X.shape[1]

In [20]:
dropout_prob = 0.02

class InksNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
        nn.Linear(input_size, 32),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(32, 64),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(64, 128),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(128, 256),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(512, 1024),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(1024, 512),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(256, 128),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.Dropout(dropout_prob),
        nn.ReLU(),
        nn.Linear(32, input_size))
    def forward(self, x):
        return self.seq(x)

### Loading

In [21]:
model = InksNet().to(device)
model.load_state_dict(torch.load(saved_models_path, map_location=torch.device('cpu')))

/tmp/ipykernel_23226/2035743707.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(saved_models_path, map_location=torch.device('cpu')))


<All keys matched successfully>

## Prediction

In [22]:
model.eval()
outputs = model(X)

## Saving result to a file

In [23]:
model_name = saved_models_path.split('/')[-1]

In [24]:
outputs_to_save = outputs.cpu().detach().numpy()

In [25]:
np.savetxt(results_path + 'prediction_for_' + which_dataset + '_dataset_from_' + model_name, outputs_to_save, delimiter=',')